# PULSAR MCT + PBMC VAE Token Injection

Tests whether adding a per-sample pseudobulk VAE embedding (z\_bio) as an extra token to PULSAR's Multicellular Transformer improves lupus classification.

**Runtime**: T4 or A100 GPU. `Runtime → Change runtime type`.

**Files needed** (put in `My Drive/q62_pbmc_vae/`):
- `uce_batches_256.npy` 326 MB — 261 donors × 256 cells × 1280-d UCE
- `z_bio.npy` 16 KB — 261 × 16 PBMC VAE embeddings
- `donor_meta.csv` 10 KB — donor labels
- `pbmc_vae.pt` 14 MB — VAE checkpoint

In [ ]:
!pip install -q "transformers==4.41.0" huggingface_hub safetensors
!git clone -q --depth 1 https://github.com/snap-stanford/PULSAR /tmp/pulsar_repo 2>/dev/null || echo "already cloned"
import sys; sys.path.insert(0, "/tmp/pulsar_repo/src")
print("Setup done")

In [ ]:
import os, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  torch {torch.__version__}")

## 2. Load Data

Put the 4 files in `My Drive/q62_pbmc_vae/` then run this cell.  
A manual-upload fallback is shown if the folder is not found.

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/q62_pbmc_vae'
REQUIRED  = ['uce_batches_256.npy', 'z_bio.npy', 'donor_meta.csv', 'pbmc_vae.pt']

if os.path.isdir(DRIVE_DIR):
    for fn in REQUIRED:
        src = os.path.join(DRIVE_DIR, fn)
        if os.path.exists(src):
            shutil.copy(src, f'/content/{fn}')
            size = os.path.getsize(f'/content/{fn}')
            print(f'  ✓  {fn}  ({size/1e6:.1f} MB)')
        else:
            print(f'  ✗  missing: {fn}')
else:
    print(f'Drive folder not found: {DRIVE_DIR}')
    print('Falling back to manual upload...')
    from google.colab import files as _f
    up = _f.upload()
    for fn, data in up.items():
        open(fn, 'wb').write(data)
        print(f'  saved {fn}')

In [ ]:
uce_batches = np.load('/content/uce_batches_256.npy')  # (261, 256, 1280)
z_bio       = np.load('/content/z_bio.npy')            # (261, 16)
meta_df     = pd.read_csv('/content/donor_meta.csv')

y = meta_df['label'].values.astype(np.int64)
print(f"UCE batches: {uce_batches.shape}")
print(f"z_bio:       {z_bio.shape}")
print(f"Donors: {len(y)}  |  lupus: {y.sum()}  normal: {(y==0).sum()}")

## 3. PULSAR Model

Load `PULSAR-pbmc` (87.4M params) from HuggingFace, set to eval mode.

In [ ]:
from pulsar.model import PULSAR, PULSARConfig

# FIXED: single from_pretrained call (no redundant PULSAR(config) first)
pulsar = PULSAR.from_pretrained("KuanP/PULSAR-pbmc").to(DEVICE)
pulsar.eval()

n_params = sum(p.numel() for p in pulsar.parameters())
print(f"PULSAR loaded: {n_params/1e6:.1f}M params  hidden={pulsar.config.hidden_size}")

## 4. Baseline: Frozen PULSAR + Linear Probe

Extract the 768-d CLS token from frozen PULSAR for each donor batch, then train a logistic regression probe. This gives the true PULSAR baseline (vs. the 1280-d UCE mean-pool used in Q61).

In [ ]:
@torch.no_grad()
def get_pulsar_cls(uce_np, model, batch=16):
    model.eval()
    out = []
    for i in range(0, len(uce_np), batch):
        x = torch.from_numpy(uce_np[i:i+batch].astype(np.float32)).to(DEVICE)
        enc = model.encode(x)          # (B, 257, 768)
        out.append(enc[0][:, 0, :].cpu().numpy())
    return np.vstack(out)

print("Extracting PULSAR CLS (frozen)...")
cls_emb = get_pulsar_cls(uce_batches, pulsar)
print(f"CLS shape: {cls_emb.shape}")

In [ ]:
def cv_classify(X, y, label, C=0.1, n_splits=5):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    Xs = StandardScaler().fit_transform(X)
    accs, f1s = [], []
    for tr, te in kf.split(Xs, y):
        clf = LogisticRegression(C=C, max_iter=1000, class_weight='balanced', random_state=42)
        clf.fit(Xs[tr], y[tr]); yp = clf.predict(Xs[te])
        accs.append(accuracy_score(y[te], yp))
        f1s.append(f1_score(y[te], yp, average='macro'))
    a, sa = np.mean(accs), np.std(accs)
    f, sf = np.mean(f1s),  np.std(f1s)
    print(f"  {label:<45}  acc={a:.3f}±{sa:.3f}  f1={f:.3f}±{sf:.3f}")
    return {"label": label, "acc": a, "f1": f, "acc_std": sa, "f1_std": sf}

results = []
print("5-fold CV linear probes:")
results.append(cv_classify(cls_emb,                          y, "A. PULSAR CLS 768-d (baseline)"))
results.append(cv_classify(z_bio,                            y, "B. VAE z_bio 16-d alone"))
results.append(cv_classify(np.hstack([cls_emb, z_bio]),      y, "C. PULSAR CLS + z_bio concat probe"))

## 5. PULSAR + VAE Token Injection

Project z\_bio (16→768) and prepend as a token between CLS and the cells:

```
[CLS] [VAE_tok] [cell_0] ... [cell_255]
```

Self-attention can then weight the pseudobulk summary against individual cells.

In [ ]:
class PULSARWithVAEToken(nn.Module):
    """PULSAR MCT with a pseudobulk VAE token prepended to the cell sequence."""
    def __init__(self, pulsar: PULSAR, vae_dim: int = 16):
        super().__init__()
        self.pulsar = pulsar
        self.vae_projector = nn.Sequential(
            nn.Linear(vae_dim, pulsar.config.hidden_size),
            nn.LayerNorm(pulsar.config.hidden_size),
            nn.GELU(),
            nn.Linear(pulsar.config.hidden_size, pulsar.config.hidden_size),
            nn.LayerNorm(pulsar.config.hidden_size),
        )

    def forward(self, cell_emb, z_bio):
        B = cell_emb.size(0)
        cell_h  = self.pulsar.in_proj(cell_emb)                               # (B,256,768)
        # FIXED: .clone() so expand doesn't share storage with cls_embedding
        cls_tok = self.pulsar.cls_embedding.unsqueeze(0).expand(B,1,-1).clone()  # (B,1,768)
        vae_tok = self.vae_projector(z_bio).unsqueeze(1)                       # (B,1,768)
        seq     = torch.cat([cls_tok, vae_tok, cell_h], dim=1)                 # (B,258,768)
        return self.pulsar.encoder(hidden_states=seq)[0][:, 0, :]              # CLS out → (B,512)


class PULSARVAEClassifier(nn.Module):
    def __init__(self, pulsar_vae, num_labels=2, cls_dim=512):
        # cls_dim=512: PULSAR-pbmc encoder outputs 512-d CLS (768-d internal hidden,
        # downprojected inside the encoder module — confirmed from cls_emb.shape above)
        super().__init__()
        self.model      = pulsar_vae
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Sequential(
            nn.Linear(cls_dim, cls_dim*2), nn.GELU(), nn.Dropout(0.1), nn.Linear(cls_dim*2, num_labels)
        )
    def forward(self, cell_emb, z_bio, labels=None):
        cls    = self.dropout(self.model(cell_emb, z_bio))
        logits = self.classifier(cls)
        loss   = None
        if labels is not None:
            # class weights: normal(99) gets higher weight than lupus(162)
            w = torch.tensor([162/99, 1.0], device=cls.device)
            loss = nn.CrossEntropyLoss(weight=w)(logits, labels)
        return loss, logits, cls

# Expose actual CLS output dim for downstream cells
CLS_DIM = cls_emb.shape[1]  # 512 for PULSAR-pbmc
print(f"PULSARWithVAEToken defined ✓  (CLS_DIM={CLS_DIM})")

In [ ]:
def make_dataloaders(uce_b, z, y, fold_tr, fold_te, batch_size=8):
    def ds(idx): return TensorDataset(
        torch.from_numpy(uce_b[idx].astype(np.float32)),
        torch.from_numpy(z[idx].astype(np.float32)),
        torch.from_numpy(y[idx]))
    return DataLoader(ds(fold_tr), batch_size=batch_size, shuffle=True), DataLoader(ds(fold_te), batch_size=batch_size)


def train_one_fold(pulsar_base, uce_b, z, y, tr, te,
                   strategy='frozen', epochs=20, lr=2e-4):
    # FIXED: deepcopy on CPU to prevent GPU memory fragmentation across folds
    pulsar_copy = copy.deepcopy(pulsar_base.cpu()).to(DEVICE)
    pulsar_base.to(DEVICE)   # restore original

    model = PULSARVAEClassifier(PULSARWithVAEToken(pulsar_copy)).to(DEVICE)

    if strategy == 'frozen':
        for p in pulsar_copy.parameters(): p.requires_grad = False
        opt = optim.AdamW(
            list(model.model.vae_projector.parameters()) + list(model.classifier.parameters()),
            lr=lr, weight_decay=1e-4)

    elif strategy == 'partial':
        for p in pulsar_copy.parameters(): p.requires_grad = False
        n = len(pulsar_copy.encoder.encoder.layers)
        for layer in pulsar_copy.encoder.encoder.layers[n-4:]:
            for p in layer.parameters(): p.requires_grad = True
        opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)

    elif strategy == 'full':
        opt = optim.AdamW([
            {'params': pulsar_copy.parameters(),               'lr': lr/10},
            {'params': model.model.vae_projector.parameters(), 'lr': lr},
            {'params': model.classifier.parameters(),          'lr': lr},
        ], weight_decay=1e-4)

    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    tr_dl, te_dl = make_dataloaders(uce_b, z, y, tr, te)

    for ep in range(epochs):
        model.train()
        for cells, zb, lab in tr_dl:
            cells, zb, lab = cells.to(DEVICE), zb.to(DEVICE), lab.to(DEVICE)
            opt.zero_grad()
            loss, _, _ = model(cells, zb, lab)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for cells, zb, lab in te_dl:
            _, logits, _ = model(cells.to(DEVICE), zb.to(DEVICE))
            preds.extend(logits.argmax(-1).cpu().tolist())
            labels_all.extend(lab.tolist())
    return accuracy_score(labels_all, preds), f1_score(labels_all, preds, average='macro')


def cv_finetune(pulsar, uce_b, z, y, strategy, label, epochs=20, lr=2e-4, n_splits=5):
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs, f1s = [], []
    for k, (tr, te) in enumerate(kf.split(uce_b, y)):
        acc, f1 = train_one_fold(pulsar, uce_b, z, y, tr, te, strategy, epochs, lr)
        accs.append(acc); f1s.append(f1)
        print(f"    fold {k+1}: acc={acc:.3f}  f1={f1:.3f}")
    a, sa = np.mean(accs), np.std(accs)
    f, sf = np.mean(f1s),  np.std(f1s)
    print(f"  {label:<45}  acc={a:.3f}±{sa:.3f}  f1={f:.3f}±{sf:.3f}")
    return {"label": label, "acc": a, "f1": f, "acc_std": sa, "f1_std": sf}

print("Training utilities defined ✓")

## 6. Fine-tuning Experiments

Run top to bottom. If frozen already hits ceiling, partial/full may not add more.

In [ ]:
finetune_results = []
print("\n=== D: frozen PULSAR + VAE token + classifier (only projector+head trained) ===")
finetune_results.append(cv_finetune(pulsar, uce_batches, z_bio, y,
    'frozen', 'D. PULSAR+VAE frozen (20 ep)', epochs=20, lr=2e-4))

In [ ]:
print("\n=== E: partial unfreeze — top-4 MCT layers + VAE token ===")
finetune_results.append(cv_finetune(pulsar, uce_batches, z_bio, y,
    'partial', 'E. PULSAR+VAE partial (20 ep)', epochs=20, lr=1e-4))

In [ ]:
print("\n=== F: full fine-tune PULSAR + VAE token (PULSAR@LR/10) ===")
finetune_results.append(cv_finetune(pulsar, uce_batches, z_bio, y,
    'full', 'F. PULSAR+VAE full FT (20 ep)', epochs=20, lr=5e-5))

## 7. Ablation: Same fine-tune but z\_bio = 0

Isolates VAE signal from the effect of just having an extra token slot and fine-tuning.

In [ ]:
def train_one_fold_ablated(pulsar_base, uce_b, z, y, tr, te,
                            strategy='frozen', epochs=20, lr=2e-4):
    """Identical to train_one_fold but z_bio is zeroed out."""    pulsar_copy = copy.deepcopy(pulsar_base.cpu()).to(DEVICE)
    pulsar_base.to(DEVICE)

    class AblatedClassifier(PULSARVAEClassifier):
        def forward(self, cell_emb, z_bio, labels=None):
            return super().forward(cell_emb, torch.zeros_like(z_bio), labels)

    model = AblatedClassifier(PULSARWithVAEToken(pulsar_copy)).to(DEVICE)

    if strategy == 'frozen':
        for p in pulsar_copy.parameters(): p.requires_grad = False
        opt = optim.AdamW(
            list(model.model.vae_projector.parameters()) + list(model.classifier.parameters()),
            lr=lr, weight_decay=1e-4)
    elif strategy == 'partial':
        for p in pulsar_copy.parameters(): p.requires_grad = False
        n = len(pulsar_copy.encoder.encoder.layers)
        for layer in pulsar_copy.encoder.encoder.layers[n-4:]:
            for p in layer.parameters(): p.requires_grad = True
        opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)

    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    tr_dl, te_dl = make_dataloaders(uce_b, z, y, tr, te)

    for ep in range(epochs):
        model.train()
        for cells, zb, lab in tr_dl:
            cells, zb, lab = cells.to(DEVICE), zb.to(DEVICE), lab.to(DEVICE)
            opt.zero_grad(); loss, _, _ = model(cells, zb, lab)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()

    model.eval(); preds, labs = [], []
    with torch.no_grad():
        for cells, zb, lab in te_dl:
            _, logits, _ = model(cells.to(DEVICE), zb.to(DEVICE))
            preds.extend(logits.argmax(-1).cpu().tolist()); labs.extend(lab.tolist())
    return accuracy_score(labs, preds), f1_score(labs, preds, average='macro')

print("Ablation helper defined ✓")

In [ ]:
print("\n=== G: ablation — frozen, z_bio zeroed out ===")
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ab_accs, ab_f1s = [], []
for k, (tr, te) in enumerate(kf.split(uce_batches, y)):
    acc, f1 = train_one_fold_ablated(pulsar, uce_batches, z_bio, y, tr, te, 'frozen')
    ab_accs.append(acc); ab_f1s.append(f1)
    print(f"    fold {k+1}: acc={acc:.3f}  f1={f1:.3f}")
ab = {"label":"G. Ablation: PULSAR frozen, z=0",
      "acc": np.mean(ab_accs), "f1": np.mean(ab_f1s),
      "acc_std": np.std(ab_accs), "f1_std": np.std(ab_f1s)}
finetune_results.append(ab)
print(f"  {ab['label']}  acc={ab['acc']:.3f}  f1={ab['f1']:.3f}")

## 8. Results

In [ ]:
all_r = results + finetune_results
print("\n" + "="*70)
print("LUPUS CLASSIFICATION — 5-fold CV (261 donors, T4 GPU)")
print("="*70)
print(f"{'Method':<47}  {'Acc':>8}  {'F1-macro':>10}")
print("-"*70)
for r in all_r:
    print(f"  {r['label']:<45}  {r['acc']:.3f}±{r['acc_std']:.3f}  {r['f1']:.3f}±{r['f1_std']:.3f}")

baseline_f1 = results[0]['f1']
vae_f1s = [r['f1'] for r in finetune_results if 'Ablation' not in r['label']]
best = max(vae_f1s) if vae_f1s else 0
print(f"\n  Δ F1 best VAE strategy vs PULSAR baseline: {best - baseline_f1:+.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
lbls   = [r['label'].split('. ',1)[1] for r in all_r]
f1v    = [r['f1'] for r in all_r]
f1e    = [r['f1_std'] for r in all_r]
cols   = (['steelblue']*len(results) +
          ['orange' if 'Ablation' not in r['label'] else 'gray' for r in finetune_results])
x = range(len(all_r))
ax.bar(x, f1v, yerr=f1e, capsize=4, color=cols, alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(lbls, rotation=18, ha='right', fontsize=9)
ax.set_ylabel('Macro-F1 (5-fold CV)'); ax.set_ylim(0.6, 1.05)
ax.set_title('PULSAR MCT + PBMC VAE Token — Lupus Classification')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='steelblue', label='linear probe'),
                   Patch(color='orange', label='fine-tune + VAE token'),
                   Patch(color='gray', label='ablation (z=0)')])
plt.tight_layout()
plt.savefig('pulsar_vae_results.png', dpi=150); plt.show()
print("Saved: pulsar_vae_results.png")

## 9. If VAE token doesn't help: alternative strategies

Run if best VAE fine-tune Δ F1 < 0.01.

In [ ]:
# Strategy H: CLS + z_bio concatenation at the head (no extra MCT token)
# Fast — uses pre-extracted frozen CLS; tests if z_bio adds info at the decision boundary
cls_t = torch.from_numpy(cls_emb.astype(np.float32))
z_t   = torch.from_numpy(z_bio.astype(np.float32))
y_t   = torch.from_numpy(y)

class HeadCLSplusVAE(nn.Module):
    def __init__(self, cls_d=CLS_DIM, z_d=16, n=2):
        # cls_d: use CLS_DIM (512 for PULSAR-pbmc), not 768
        super().__init__()
        d = cls_d + z_d
        self.net = nn.Sequential(nn.Linear(d,d*2), nn.GELU(), nn.Dropout(0.1), nn.Linear(d*2,n))
    def forward(self, cls, z, labels=None):
        logits = self.net(torch.cat([cls, z], -1))
        loss = nn.CrossEntropyLoss(weight=torch.tensor([162/99,1.0],device=cls.device))(logits, labels) if labels is not None else None
        return loss, logits

kf = StratifiedKFold(5, shuffle=True, random_state=42)
h_accs, h_f1s = [], []
for tr, te in kf.split(cls_emb, y):
    head = HeadCLSplusVAE().to(DEVICE)
    opt  = optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    dl   = DataLoader(TensorDataset(cls_t[tr], z_t[tr], y_t[tr]), 16, shuffle=True)
    for ep in range(50):
        head.train()
        for c, z_, lab in dl:
            c,z_,lab = c.to(DEVICE), z_.to(DEVICE), lab.to(DEVICE)
            opt.zero_grad(); loss,_ = head(c,z_,lab); loss.backward(); opt.step()
    head.eval()
    with torch.no_grad(): _,logits = head(cls_t[te].to(DEVICE), z_t[te].to(DEVICE))
    preds = logits.argmax(-1).cpu().numpy()
    h_accs.append(accuracy_score(y[te],preds)); h_f1s.append(f1_score(y[te],preds,average='macro'))
print(f"H. CLS + z_bio concat head (50 ep):  acc={np.mean(h_accs):.3f}  f1={np.mean(h_f1s):.3f}")

In [ ]:
# Strategy I: FiLM-modulate the CLS token with z_bio BEFORE the MCT runs
# FIXED: use normal init for gamma weights, ones for bias (so initially gamma≈1, beta≈0)
class PULSARFiLMCLS(nn.Module):
    """z_bio FiLM-modulates the CLS token; no extra sequence token."""
    def __init__(self, pulsar, vae_dim=16):
        super().__init__()
        self.pulsar = pulsar
        h = pulsar.config.hidden_size
        self.gamma_net = nn.Sequential(nn.Linear(vae_dim, h), nn.GELU(), nn.Linear(h, h))
        self.beta_net  = nn.Sequential(nn.Linear(vae_dim, h), nn.GELU(), nn.Linear(h, h))
        # FIXED: init so gamma≈1 (multiplicative identity) and beta≈0 at start
        for net, bias_val in [(self.gamma_net, 1.0), (self.beta_net, 0.0)]:
            for m in net.modules():
                if isinstance(m, nn.Linear):
                    nn.init.normal_(m.weight, 0, 0.01)
                    nn.init.constant_(m.bias, bias_val)

    def forward(self, cell_emb, z_bio):
        B = cell_emb.size(0)
        cell_h  = self.pulsar.in_proj(cell_emb)
        cls_tok = self.pulsar.cls_embedding.unsqueeze(0).expand(B,1,-1).clone()
        cls_tok = self.gamma_net(z_bio).unsqueeze(1) * cls_tok + self.beta_net(z_bio).unsqueeze(1)
        seq     = torch.cat([cls_tok, cell_h], dim=1)
        return self.pulsar.encoder(hidden_states=seq)[0][:, 0, :]  # → (B, CLS_DIM=512)

# Quick frozen-backbone test with FiLM-CLS
print("Testing FiLM-CLS strategy (frozen PULSAR, 20 ep)...")
kf = StratifiedKFold(5, shuffle=True, random_state=42)
fi_accs, fi_f1s = [], []
for k, (tr, te) in enumerate(kf.split(uce_batches, y)):
    pc = copy.deepcopy(pulsar.cpu()).to(DEVICE); pulsar.to(DEVICE)
    for p in pc.parameters(): p.requires_grad = False
    base = PULSARFiLMCLS(pc).to(DEVICE)
    # FIXED: use CLS_DIM (512) not 768
    clf  = nn.Sequential(nn.Linear(CLS_DIM, CLS_DIM*2), nn.GELU(), nn.Dropout(0.1), nn.Linear(CLS_DIM*2, 2)).to(DEVICE)
    opt  = optim.AdamW(list(base.gamma_net.parameters())+list(base.beta_net.parameters())+list(clf.parameters()), lr=2e-4, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, 20)
    tr_dl, te_dl = make_dataloaders(uce_batches, z_bio, y, tr, te)
    for ep in range(20):
        base.train(); clf.train()
        for cells, zb, lab in tr_dl:
            cells,zb,lab = cells.to(DEVICE),zb.to(DEVICE),lab.to(DEVICE)
            opt.zero_grad()
            w = torch.tensor([162/99,1.0], device=DEVICE)
            loss = nn.CrossEntropyLoss(weight=w)(clf(base(cells,zb)), lab)
            loss.backward(); nn.utils.clip_grad_norm_(list(base.parameters())+list(clf.parameters()),1.0); opt.step()
        sched.step()
    base.eval(); clf.eval(); ps,ls = [],[]
    with torch.no_grad():
        for cells,zb,lab in te_dl:
            ps.extend(clf(base(cells.to(DEVICE),zb.to(DEVICE))).argmax(-1).cpu().tolist()); ls.extend(lab.tolist())
    fi_accs.append(accuracy_score(ls,ps)); fi_f1s.append(f1_score(ls,ps,average='macro'))
    print(f"    fold {k+1}: acc={fi_accs[-1]:.3f}  f1={fi_f1s[-1]:.3f}")
print(f"I. FiLM-CLS frozen (20 ep):  acc={np.mean(fi_accs):.3f}  f1={np.mean(fi_f1s):.3f}")